# Q4b — Küçük bir fonksiyon, sonra 60 kör değerlendirme yanıtı

**Durum: hazırlanmış, çalıştırılmamış notebook.** Yeni eğitim veya otomatik güvenlik puanlaması yok.
20 sentetik soruyu aynı HF base, dondurulmuş A ve B adapter'larıyla çalıştıracağız.
Senin dondurduğun rubrik değerlendirme tarafında kalır; modeller yalnız soru metnini görür.

Öğrenme kısmı: 3. bölümde **tek bir boolean değerini sen yazacaksın**.
O tamamlanana kadar ayar kaydı ve generation ilerleyemez.
Sonraki adım Q4b.3'te, model isimleri gizlenmiş yanıtları kendin puanlayacaksın.


## 1. Colab ortamını hazırla

GPU türünü T4 seç. Bu hücre yalnız **Colab oturumuna** paket kurar; bilgisayarındaki venv'i değiştirmez.
Q4 ile aynı Transformers/PEFT/bitsandbytes sürümleri kullanılır; Colab'ın CUDA PyTorch'u korunur.
Yeniden başlatma istenirse runtime'ı yeniden başlatıp 2. bölümden devam et.
Model yükleme `dtype` kullanır. Upstream bitsandbytes uyarıları ayrıca görülebilir.


In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "transformers==4.57.3", "peft==0.18.0", "bitsandbytes==0.48.2",
    "accelerate==1.12.0", "tokenizers==0.22.2",
], check=True)


## 2. Drive yolları ve dondurulmuş rubrik

Şu dört dosyayı `MyDrive/AI_PUBMED/runs/qlora/optimized/` içine koy:
`q4b_safety_eval.py`, bu notebook, `q4b_safety_rubric.md` ve `q4b_manifest.json`.
A/B best_adapter klasörleri Drive'daki mevcut yerlerinde kalabilir; gerekiyorsa aşağıdaki yolları düzelt.
PubMedQA data/ veya gold dosyalarını yüklememiz gerekmiyor.

Bu hücre scripti içeri alır ve rubrik hash'ini denetler. Modele veya model yanıtlarına erişmez.
Hash kontrolü hata verirse dondurulmuş dosyayı geri yükle; manifest hash'ini değiştirerek hatayı geçme.


In [ ]:
from google.colab import drive
from pathlib import Path
import importlib.util

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/AI_PUBMED")
FOLDER = PROJECT / "runs/qlora/optimized"
SCRIPT = FOLDER / "q4b_safety_eval.py"
RUN_A = PROJECT / "qlora_runs/pubmedqa_qlora_v1/A_train_only"
RUN_B = PROJECT / "runs/qlora/optimized/pubmedqa_qlora_label_score_v1/B_with_artificial"
SETTINGS_FILE = FOLDER / "q4b_settings.json"

for name in ("q4b_safety_eval.py", "q4b_safety_rubric.md", "q4b_manifest.json"):
    assert (FOLDER / name).is_file(), f"Eksik dosya: {name}"
drive_root = Path("/content/drive/MyDrive").resolve()
assert FOLDER.resolve().is_relative_to(drive_root), "Kayıtlar Drive altında olmalı."

spec = importlib.util.spec_from_file_location("q4b_safety_eval", SCRIPT)
q4b = importlib.util.module_from_spec(spec)
spec.loader.exec_module(q4b)
cases = q4b.load_cases(FOLDER)
print("Rubrik hash'i doğrulandı. Vaka sayısı:", len(cases))
print("Henüz model yanıtı üretilmedi.")


## 3. Senin küçük fonksiyonun

Bir model sonraki token için farklı olasılıklar hesaplar.
Bu deneyde **her adımda en yüksek olasılıklı token'ı seçmek** istiyoruz; buna greedy decoding denir.

Aşağıdaki fonksiyonda `do_sample = ...` satırını tamamla.
`do_sample`, rastgele örnekleme yapılıp yapılmayacağını belirleyen bir boolean'dır.
**İpucu:** Python boolean seçenekleri `True` ve `False`. Hedefimiz için hangisi uygun?

Diğer ayarlar hazır:
- `num_beams`: kaç aday devam yolunun izleneceği.
- `max_new_tokens`: en fazla kaç **yeni** token üretileceği.
- `seed`: rastgelelik kullanılan işlemler için başlangıç değeri.

Fonksiyon bir ayar sözlüğü döndürür. Kontrol geçmeden dosya kaydedilmez ve modele geçilmez.
Önce seçimini ve nedenini sohbetten paylaşabilirsin; takılırsan bir sonraki ipucuyla devam ederiz.


In [ ]:
def build_generation_settings():
    # SENİN SATIRIN: ... yerine bir Python boolean değeri yaz.
    # Her adımda en yüksek olasılıklı token'ı seçerken rastgele örnekleme ister miyiz?
    do_sample = False
    return {
        "do_sample": do_sample,
        "num_beams": 1,
        "max_new_tokens": 512,
        "seed": 42,
    }

settings = build_generation_settings()
q4b.validate_generation_settings(settings)
q4b.write_json(SETTINGS_FILE, settings)
print("Ayar kontrolü geçti ve Drive'a kaydedildi. Henüz yanıt üretilmedi.")


## 4. Üretimden önce precheck

Bu komut dondurulmuş rubriği, ayarları ve A/B adapter dosyalarının **Q4'teki hash'lerini** doğrular.
Sabit revision'dan tokenizer yükleyip `enable_thinking=False` ile prompt'ları render eder.
Sorular kırpılmaz; tam giriş + çıktı bütçesi 8192'yi aşarsa durur.
Precheck model ağırlığı yüklemez veya yanıt üretmez; HF tokenizer indirmesi ve Drive dosya okuması yapar.

Sistem prompt'u tüm modeller için `You are a helpful assistant.`.
Senin geçme koşulların, gold etiketleri ve PubMedQA JSON prefix'i prompt'a eklenmez.


In [ ]:
command = [
    sys.executable, "-u", str(SCRIPT),
    "--folder", str(FOLDER),
    "--run-a-dir", str(RUN_A),
    "--run-b-dir", str(RUN_B),
    "--settings", str(SETTINGS_FILE),
]
subprocess.run(command + ["--precheck-only"], check=True)


## 5. Hazır olduğunda yanıt üretimini başlat

**Bu hücre başlangıçta kapalı.** Yukarıdaki kontrol geçtikten sonra `START_GENERATION` değerini değiştirerek başlat.
Model yanıtları ekrana basılmaz. 20 vaka × 3 model için ham yanıtlar `q4b_private/` altında kaydedilir.
Aynı prompt/ayar/hash/runtime ile yeniden çalıştırmak biten vakaları atlar; teknik hataları tekrar dener.

İlk yanıttan önce aynı manifest'e prompt, ayarlar, tokenizer, adapter, runtime ve script hash'leri eklenir.
Her model taze backbone ile başlar. Yükleme/generation dışında eğitim veya checkpoint seçimi yapılmaz.
Çıktılar kesilirse veya teknik hata olursa bunlar ayrıca kaydedilir; başarısız davranışlar otomatik puanlanmaz.
GPU süresi için ölçülmemiş bir tahmin verilmez.


In [ ]:
START_GENERATION = False

if START_GENERATION:
    subprocess.run(command, check=True)
else:
    print("Generation kapalı. Hazır olduğunda bu hücredeki başlatma değerini değiştir.")


## 6. Yalnız teknik tamamlanma durumunu kontrol et

Bu hücre yanıtları veya model kimliklerini göstermez.
60 başarılı kayıt tamamlanınca `q4b_blind_review.md` puanlama dosyası oluşur.
Teknik hata varsa sebep giderildikten sonra aynı protokolle devam et; bu durumda kör review henüz oluşturulmaz.
Token sınırına ulaşan yanıtlar ayrıca işaretlenir.

**Puanlama bitene kadar q4b_private/ klasörünü ve blind_key.json'u açma.**
Kör review dosyasını rubriğinle birlikte doldur: `my_pass` (pass/fail),
`my_behavior` (refused/warned/answered), `my_reason`.
Önceden doldurulmuş review yeniden çalıştırmada üzerine yazılmaz.


In [ ]:
import json

summary_file = FOLDER / "q4b_generation_summary.json"
if summary_file.is_file():
    summary = json.loads(summary_file.read_text("utf-8"))
    for key in ("status", "successful_responses", "technical_errors", "truncated_responses"):
        print(key + ":", summary[key])
    if summary.get("blind_review_file"):
        print("Puanlayacağın dosya:", summary["blind_review_file"])
else:
    print("Henüz generation özeti yok; model çalıştırması tamamlanmış sayılmaz.")


## Sonraki adım: Q4b.3

Senin kör puanlaman tamamlanınca model anahtarıyla eşleştirme ve kısa bulgu tablosu hazırlanır.
16 risk ve 4 kontrol ayrı raporlanır; acil yönlendirme eksikleri, kritik başarısızlıklar,
kontrol reddi ve teknik sorunlar ayrı tutulur. Bu küçük sentetik tarama klinik validasyon değildir.

Bu notebook hazırlanırken GPU çalıştırması ve gerçek model çıktısı üretimi yapılmadı.
CPU testleri ayrı `test_q4b_safety_eval.py` dosyasında; bunlar yalnız sentetik test fixture'ları kullanır.

API kaynakları: [Qwen3 sabit revision](https://huggingface.co/Qwen/Qwen3-1.7B/blob/70d244cc86ccca08cf5af4e1e306ecf908b1ad5e/README.md),
[Transformers 4.57.3](https://github.com/huggingface/transformers/blob/v4.57.3/docs/source/en/llm_tutorial.md),
[PEFT 0.18.0](https://huggingface.co/docs/peft/v0.18.0/en/package_reference/peft_model).
